In [ ]:
from brails.types import RegionBoundary

region_boundary_object = RegionBoundary(
    {
        "type": "locationName",
        "data": "Berkeley, CA"
    }
)

geometry, description, osm_id = region_boundary_object.get_boundary()
display(geometry)

In [ ]:
from brails.scrapers import OvertureMapsFootprintScraper

bldg_inventory = OvertureMapsFootprintScraper({'length': 'ft'}).get_footprints(region_boundary_object)

# only use the footprint geometry and remove all other metadata for clarity
bldg_inventory.remove_features(bldg_inventory.get_all_asset_features())

In [ ]:
from brails.scrapers import NSI_Parser

nsi_points = NSI_Parser().get_raw_data(region_boundary_object)

# Rename the features we need later using standard SimCenter labels
feature_rename_map = {
    'fd_id': 'fd_id',
    'type': 'type',
    'bldgtype': 'BuildingType',
#    'found_type': 'FoundationType',
    'found_ht': 'FirstFloorElevation',
    'pop2amu65': 'NightPopulationUnder65',
    'pop2amo65': 'NightPopulationOver65',
    'pop2pmu65': 'DayPopulationUnder65',
    'pop2pmo65': 'DayPopulationOver65',
    'x': 'Longitude',
    'y': 'Latitude',
    'sqft': 'PlanArea',
    'num_story': 'NumberOfStories',
    'students': 'StudentPopulation',
    'med_yr_blt': 'YearBuilt',
    'occtype': 'OccupancyClass'
}
nsi_points.change_feature_names(feature_rename_map)

# remove all other features for clarity
features_to_remove = nsi_points.get_all_asset_features()
features_to_remove.difference_update(feature_rename_map.values())
nsi_points.remove_features(features_to_remove)

In [ ]:
from brails.aggregators import BasicPointsToPolygonsAllocator

BasicPointsToPolygonsAllocator(
    polygon_inventory=bldg_inventory,
    point_inventory=nsi_points,
).allocate(
    use_convex_hull = True,
    buffer_dist = 10.0
)

In [ ]:
from brails.types import AssetInventory
from copy import deepcopy
import pandas as pd
import random

def fix_invalid_bldg_configs(inventory: AssetInventory):

    bldg_inventory = deepcopy(inventory)

    issues_dict = {}

    for asset_id, asset in bldg_inventory.inventory.items():
        stories = asset.features.get('NumberOfStories', None)
        occupancy = asset.features.get('OccupancyClass', None)
        year = asset.features.get('YearBuilt', None)

        # if any of the key features are missing, skip this asset
        if stories is None or occupancy is None or year is None:
            continue

        # map occupancy classes to the strict Hazus names
        if occupancy[:4] == 'RES3':
            asset.add_features({
                'OccupancyClassNSI': occupancy,
                'OccupancyClass': occupancy[:5]
            })
        else:
            asset.add_features({
                'OccupancyClassNSI': occupancy,
                'OccupancyClass': occupancy[:4]
            })

        stories = int(stories)
        year = int(year)

        flagged = False

        if stories >= 7 and occupancy not in [
            'RES3A', 'RES3B', 'RES3C', 'RES3D', 'RES3E', 'RES3F',
            'RES4', 'RES5',
            'COM4', 'COM5', 'COM6',
            'GOV1',
            'EDU2'
        ]:

            flagged = True

            if occupancy.startswith('RES'):
                asset.add_features(
                    {
                        'OccupancyClass': 'RES3F',
                        'OccupancyClassNSI': 'RES3F'
                    },
                    overwrite=True
                )

            elif occupancy.startswith('COM'):
                occupancy_to_use = random.choice(['COM4', 'COM5', 'COM6'])
                asset.add_features(
                    {
                        'OccupancyClass': occupancy_to_use,
                        'OccupancyClassNSI': occupancy_to_use
                    },
                    overwrite=True
                )

            elif occupancy.startswith('GOV'):
                asset.add_features(
                    {
                        'OccupancyClass': 'GOV1',
                        'OccupancyClassNSI': 'GOV1'
                    },
                    overwrite=True
                )

            elif occupancy.startswith('EDU'):
                asset.add_features(
                    {
                        'OccupancyClass': 'EDU2',
                        'OccupancyClassNSI': 'EDU2'
                    },
                    overwrite=True
                )

            elif occupancy.startswith('IND'):
                occupancy_to_use = random.choice(['COM4', 'COM5', 'COM6'])
                asset.add_features(
                    {
                        'OccupancyClass': occupancy_to_use,
                        'OccupancyClassNSI': occupancy_to_use
                    },
                    overwrite=True
                )

            elif occupancy.startswith('REL'):
                asset.add_features(
                    {
                        'OccupancyClass': 'GOV1',
                        'OccupancyClassNSI': 'GOV1'
                    },
                    overwrite=True
                )

            else:
                print(f'UNEXPECTED OCCUPANCY CLASS for high rise: {occupancy}')

        elif stories >= 4:

            if year < 1950 and occupancy in ['RES1', 'RES2', 'IND1', 'EDU1', 'GOV2']:
                flagged = True

            elif year < 1970 and occupancy in ['RES1', 'RES2', 'IND6', 'EDU1']:
                flagged = True

            elif occupancy in ['RES1', 'RES2', 'IND1', 'IND2', 'IND6', 'EDU1']:
                flagged = True

            if flagged:

                if occupancy.startswith('RES'):
                    asset.add_features(
                        {
                            'OccupancyClass': 'RES3C',
                            'OccupancyClassNSI': 'RES3C'
                        },
                        overwrite=True
                    )

                elif occupancy.startswith('GOV'):
                    asset.add_features(
                        {
                            'OccupancyClass': 'GOV1',
                            'OccupancyClassNSI': 'GOV1'
                        },
                        overwrite=True
                    )

                elif occupancy.startswith('EDU'):
                    asset.add_features(
                        {
                            'OccupancyClass': 'EDU2',
                            'OccupancyClassNSI': 'EDU2'
                        },
                        overwrite=True
                    )

                elif occupancy.startswith('IND'):
                    occupancy_to_use = random.choice(['IND3', 'IND4', 'IND5'])
                    asset.add_features(
                        {
                            'OccupancyClass': occupancy_to_use,
                            'OccupancyClassNSI': occupancy_to_use
                        },
                        overwrite=True
                    )

                else:
                    print(f'UNEXPECTED OCCUPANCY CLASS for mid rise: {occupancy}')


        if flagged:
            issues_dict.update({asset_id: {
                'NumberOfStories': stories,
                'OccupancyClass': occupancy,
                'YearBuilt': year}}
            )

    return bldg_inventory, pd.DataFrame(issues_dict).T

In [ ]:
bldg_inventory_fixed, issues_df = fix_invalid_bldg_configs(bldg_inventory)

issues_df

In [ ]:
from brails.imputers import KnnImputer

bldg_inventory_imputed = KnnImputer(
    bldg_inventory_fixed,
    n_possible_worlds=1,
    exclude_features=['Latitude','Longitude','fd_id','id']
).impute()

In [ ]:
bldg_inventory_imputed_fixed, issues_df = fix_invalid_bldg_configs(bldg_inventory_imputed)

issues_df

In [ ]:
from brails.inferers import HazusInfererEarthquake

# add features shared by all buildings
for asset_id, asset in bldg_inventory_imputed_fixed.inventory.items():
    asset.add_features({
        'Income': 78672,     # This could be based on Households assigned to this building
        'SplitLevel': "No",  # This could be smarter since we know split level from NSI
    })

bldg_inventory_final = HazusInfererEarthquake(
    input_inventory=bldg_inventory_imputed_fixed,
    clean_features=False
).infer()

In [ ]:
for asset_id, asset in bldg_inventory_final.inventory.items():

    # we have to adjust the RES3 OccupancyClasses
    occupancy = asset.features.get('OccupancyClass', None)

    # map occupancy classes to the strict Hazus names
    if occupancy[:4] == 'RES3':
        asset.add_features({
            'OccupancyClass': occupancy[:4]
        })

    # and we have to make sure certain structural systems are not better than Low-code
    if (
        asset.features.get('StructureType', None) in ['S5','C3','URM'] and
        asset.features.get('DesignLevel', None) in ['Moderate-Code', 'High-Code']
    ):
        asset.add_features({
            'DesignLevel': 'Low-Code'
        })


In [ ]:
# Buildings
bldg_inventory_final.write_to_geojson('bldg_inventory_EQ.geojson')

In [ ]:
# to get a CSV
import geopandas as gpd

bldg_geojson = bldg_inventory_final.get_geojson()
bldg_gdf = gpd.GeoDataFrame.from_features(
    bldg_geojson['features'],
    crs = 'EPSG:4326'
)

centroids = bldg_gdf.geometry.centroid
bldg_gdf['Longitude'] = centroids.x
bldg_gdf['Latitude'] = centroids.y

bldg_df = bldg_gdf.drop(columns=['geometry'])

bldg_df.to_csv('bldg_inventory_EQ.csv')